# Etapa 1 - Análise exploratória (EDA)

Este notebook carrega a base Bank Marketing, verifica sua qualidade, explora a variável alvo e documenta o tratamento inicial dos dados.

A coluna `duration` é removida na preparação porque representa uma informação conhecida somente depois do contato e causaria vazamento temporal.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'src' / 'utils.py').exists():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do projeto.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_FILE
from src.utils import clean_and_prepare_data, load_data, print_data_summary

sns.set_theme(style='whitegrid')
print(f'Raiz do projeto: {PROJECT_ROOT}')
print(f'Arquivo de dados: {DATA_FILE}')

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f'Base nao encontrada em {DATA_FILE}. Baixe a base Kaggle Bank Marketing e coloque-a nesse caminho.'
    )

df = load_data(str(DATA_FILE))
print(f'Dimensoes: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
display(df.head())

## Qualidade e estrutura dos dados

In [ ]:
quality_summary = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'unicos': df.nunique(),
}).sort_values('nulos', ascending=False)

print(f'Duplicatas: {df.duplicated().sum():,}')
display(quality_summary)
display(df.describe(include='all').T)

## Variável alvo e distribuições principais

In [ ]:
target_summary = (
    df['y'].value_counts(dropna=False)
    .rename_axis('target')
    .reset_index(name='quantidade')
)
target_summary['proporcao'] = target_summary['quantidade'] / len(df)
display(target_summary)
print(f'Taxa de conversao: {(df["y"] == "yes").mean():.2%}')

In [ ]:
visualizations_dir = PROJECT_ROOT / 'visualizations'
visualizations_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x='y', ax=axes[0], order=['no', 'yes'])
axes[0].set_title('Distribuicao da conversao')
axes[0].set_xlabel('Adesao ao deposito')
axes[0].set_ylabel('Quantidade')

sns.histplot(data=df, x='age', hue='y', bins=30, element='step', stat='density', common_norm=False, ax=axes[1])
axes[1].set_title('Distribuicao de idade por resultado')
axes[1].set_xlabel('Idade')
axes[1].set_ylabel('Densidade')

fig.tight_layout()
fig.savefig(visualizations_dir / 'eda_target_age.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
job_counts = df['job'].value_counts().head(10).sort_values()
conversion_by_job = (
    df.assign(converted=(df['y'] == 'yes').astype(int))
    .groupby('job')['converted']
    .mean()
    .sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
job_counts.plot.barh(ax=axes[0], color='#F8B500')
axes[0].set_title('Principais categorias de emprego')
axes[0].set_xlabel('Quantidade')

conversion_by_job.plot.barh(ax=axes[1], color='#4ECDC4')
axes[1].set_title('Taxa de conversao por emprego')
axes[1].set_xlabel('Taxa de conversao')

fig.tight_layout()
fig.savefig(visualizations_dir / 'eda_job_conversion.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x='balance', hue='y', bins=50, element='step', stat='density', common_norm=False, ax=axes[0])
axes[0].set_title('Distribuicao de saldo por resultado')
axes[0].set_xlabel('Saldo medio anual')

sns.boxplot(data=df, x='y', y='balance', order=['no', 'yes'], ax=axes[1])
axes[1].set_title('Saldo por resultado')
axes[1].set_xlabel('Adesao ao deposito')

fig.tight_layout()
fig.savefig(visualizations_dir / 'eda_balance.png', dpi=120, bbox_inches='tight')
plt.show()

## Tratamento inicial

A função compartilhada do projeto remove duplicatas, descarta linhas com valores ausentes e exclui `duration`. A codificação e a normalização para o modelo são executadas no notebook da Etapa 2.

In [ ]:
df_clean = clean_and_prepare_data(df, drop_duration=True)
removed_columns = sorted(set(df.columns) - set(df_clean.columns))

print(f'Linhas antes: {len(df):,}')
print(f'Linhas depois: {len(df_clean):,}')
print(f'Colunas removidas: {removed_columns}')
print(f'Coluna duration presente apos o tratamento? {"duration" in df_clean.columns}')
display(df_clean.head())
print_data_summary(df_clean)

## Conclusões da EDA

- A variável `y` é binária e apresenta desbalanceamento, portanto a taxa de conversão deve ser reportada explicitamente.
- As variáveis de cliente e de campanha estão disponíveis antes da decisão e podem compor o contexto.
- `duration` foi removida para evitar vazamento temporal.
- A preparacão final, a comparação com o baseline e o Thompson Sampling estão no notebook `02_modelo.ipynb`.